# Project: RNN vs LSTM on the adding problem ➕
The **adding problem** (Hochreiter & Schmidhuber 1997) is the classic test of long-term memory: a long sequence of random numbers, two of them marked; output their sum. A plain RNN must carry the first marked number across dozens of steps through repeated multiplications, so its gradients vanish. An **LSTM**'s additive cell state is a highway for information. Write the LSTM cell yourself, then race the two.

Topic: [[RNN and LSTM]], [[Backpropagation]] · guide note: [[Project - RNN vs LSTM on the Adding Problem]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import time, numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); torch.set_num_threads(4)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

## 1. The LSTM cell
Four gates from one matrix multiply, in PyTorch's order **i, f, g, o** (input, forget, candidate, output):
$i,f,o = \sigma(\cdot)$, $g = \tanh(\cdot)$, $c' = f\odot c + i\odot g$, $h' = o\odot\tanh(c')$, where $(\cdot) = W_xx + W_hh + b$.

In [ ]:
def lstm_cell(x, h, c, W_x, W_h, bias):
    # TODO 1: gates, new cell state, new hidden state → (h_new, c_new)
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_cell():
    torch.manual_seed(0); cell = nn.LSTMCell(3, 5); x, h, c = torch.randn(4, 3), torch.randn(4, 5), torch.randn(4, 5)
    ref_h, ref_c = cell(x, (h, c))
    my_h, my_c = lstm_cell(x, h, c, cell.weight_ih, cell.weight_hh, cell.bias_ih + cell.bias_hh)
    return torch.allclose(my_h, ref_h, atol=1e-6) and torch.allclose(my_c, ref_c, atol=1e-6)
check("LSTM cell", _t_cell, "Must match torch.nn.LSTMCell exactly (gate order i, f, g, o).")

## 2. The data
Return `x` of shape `(n, T, 2)`: channel 0 = uniform random values, channel 1 = markers (exactly two 1s: one in the first half, one in the second half), and `y` of shape `(n, 1)` = the sum of the two marked values. Use `gen` for the marker positions.

In [ ]:
def adding_problem(n, T, gen):
    # TODO 2: random values, two markers (one per half), target = sum of the marked values
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_data():
    g = torch.Generator().manual_seed(0); x, y = adding_problem(50, 20, g)
    return x.shape == (50, 20, 2) and y.shape == (50, 1) and (x[..., 1].sum(1) == 2).all() and (x[:, :10, 1].sum(1) == 1).all() and torch.allclose((x[..., 0] * x[..., 1]).sum(1, keepdim=True), y)
check("adding data", _t_data, "Shapes (n, T, 2) and (n, 1); one marker in each half; y = sum of the marked values.")

class SeqModel(nn.Module):
    def __init__(self, kind, hidden=32):
        super().__init__(); self.rnn = (nn.LSTM if kind == "LSTM" else nn.RNN)(2, hidden, batch_first=True); self.out = nn.Linear(hidden, 1)
    def forward(self, x): return self.out(self.rnn(x)[0][:, -1])

def train(kind, T, steps=1500, seed=0):
    torch.manual_seed(seed); g = torch.Generator().manual_seed(seed); m = SeqModel(kind); opt = torch.optim.Adam(m.parameters(), 3e-3)
    Xv, Yv = adding_problem(500, T, g); curve = []
    for s in range(steps):
        x, y = adding_problem(64, T, g); loss = F.mse_loss(m(x), y); opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        if s % 100 == 99:
            with torch.no_grad(): curve.append(F.mse_loss(m(Xv), Yv).item())
    return curve

results = {}
if ready("adding data"):
    print("baseline (always predict 1.0): MSE = Var(sum of two U(0,1)) = 1/6 ≈ 0.167")
    for T in (10, 50):
        for kind in ("RNN", "LSTM"):
            t = time.time(); results[(kind, T)] = train(kind, T); print(f"T={T:<3} {kind:<4} final validation MSE {results[(kind, T)][-1]:.4f}  ({time.time() - t:.0f}s)")
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.4), sharey=True)
    for a, T in zip(ax, (10, 50)):
        for kind in ("RNN", "LSTM"): a.plot(range(100, 1501, 100), results[(kind, T)], "o-", label=kind)
        a.axhline(1 / 6, ls="--", c="gray", label="baseline"); a.set_title(f"sequence length {T}"); a.set_xlabel("step"); a.legend()
    ax[0].set_ylabel("validation MSE"); plt.show()
    check("LSTM remembers", lambda: results[("LSTM", 50)][-1] < 0.05 and results[("RNN", 50)][-1] > 2 * results[("LSTM", 50)][-1],
          "At T = 50 the LSTM should get far below the 0.167 baseline while the plain RNN lags behind.")

<details><summary>▶ Solution</summary>

```python
def lstm_cell(x, h, c, W_x, W_h, bias):
    gates = x @ W_x.T + h @ W_h.T + bias
    i, f, g, o = gates.chunk(4, dim=-1)
    i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
    g = torch.tanh(g)
    c_new = f * c + i * g
    h_new = o * torch.tanh(c_new)
    return h_new, c_new
```

```python
def adding_problem(n, T, gen):
    vals = torch.rand(n, T)
    marks = torch.zeros(n, T)
    for row in range(n):
        a = int(torch.randint(0, T // 2, (1,), generator=gen)); b_ = int(torch.randint(T // 2, T, (1,), generator=gen))
        marks[row, a] = marks[row, b_] = 1
    x = torch.stack([vals, marks], dim=-1)
    y = (vals * marks).sum(1, keepdim=True)
    return x, y
```
</details>

## Stretch goals
- Push T to 100 and 200. Where does the LSTM start to struggle?
- Add a GRU (`nn.GRU`) to the race.
- Plot the gradient norm of the loss w.r.t. the *first* input for RNN vs LSTM: vanishing gradients, measured.